In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / 'common' / 'scripts'))
from metabase_client import MetabaseClient

# ── Parameters ──
NAV_DATE = '2026-03-04'

FUNDS = [
    {'code': 'TUK75', 'name': 'Tuleva Maailma Aktsiate Pensionifond', 'isin': 'EE3600109435'},
    {'code': 'TUK00', 'name': 'Tuleva Maailma Võlakirjade Pensionifond', 'isin': 'EE3600109443'},
]

client = MetabaseClient()
print(f'NAV date: {NAV_DATE}')

NAV date: 2026-03-04


In [2]:
# ── Fetch all data from Metabase ──
raw_positions = pd.DataFrame(client.execute_card(2296))
raw_index = pd.DataFrame(client.execute_card(2297))
raw_units = pd.DataFrame(client.execute_card(2298))
raw_registry = pd.DataFrame(client.execute_card(2299))

print(f'Card 2296 (positions):  {len(raw_positions)} rows')
print(f'Card 2297 (index):      {len(raw_index)} rows')
print(f'Card 2298 (units):      {len(raw_units)} rows')
print(f'Card 2299 (registry):   {len(raw_registry)} rows')

# ── Prepare per-fund data ──
fund_data = {}
for fund in FUNDS:
    fd = {}
    code = fund['code']

    all_positions = raw_positions[
        (raw_positions['Fund Code'] == code) &
        (raw_positions['Nav Date'] == NAV_DATE)
    ].copy()

    fd['all_positions'] = all_positions
    fd['securities'] = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
    fd['cash'] = all_positions[all_positions['Account Type'] == 'CASH'].copy()
    fd['receivables'] = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
    fd['liabilities'] = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
    fd['units_row'] = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

    print(f'\n{code} on {NAV_DATE}: {len(all_positions)} rows  '
          f'({len(fd["securities"])} sec, {len(fd["cash"])} cash, '
          f'{len(fd["receivables"])} recv, {len(fd["liabilities"])} liab, {len(fd["units_row"])} units)')

    # Units data
    all_fund_units = raw_units[raw_units['Security Name'] == fund['name']].copy()
    all_fund_units = all_fund_units.sort_values('Request Date', ascending=False)
    fd['units_today'] = all_fund_units[all_fund_units['Request Date'] == NAV_DATE]

    prev_dates = all_fund_units[all_fund_units['Request Date'] < NAV_DATE]['Request Date'].unique()
    prev_date = None
    if len(prev_dates) > 0 and len(fd['units_today']) > 0:
        today_nav_val = fd['units_today'].iloc[0]['Nav']
        for d in sorted(prev_dates, reverse=True):
            row = all_fund_units[all_fund_units['Request Date'] == d].iloc[0]
            if row['Nav'] != today_nav_val:
                prev_date = d
                break
        if prev_date is None:
            prev_date = sorted(prev_dates)[-1]
    fd['prev_date'] = prev_date
    fd['units_prev'] = all_fund_units[all_fund_units['Request Date'] == prev_date] if prev_date else pd.DataFrame()
    print(f'  Units dates: today={NAV_DATE}, prev={prev_date}')

    fund_reg = raw_registry[raw_registry['Isin'] == fund['isin']]
    if len(fund_reg) > 0:
        fd['mgmt_fee_rate'] = fund_reg.iloc[0]['Management Fee Rate']
        print(f'  Mgmt fee: {fd["mgmt_fee_rate"]*100:.3f}% p.a.')
    else:
        fd['mgmt_fee_rate'] = None
        print(f'  WARNING: Fund not found in registry!')

    fund_data[code] = fd

Card 2296 (positions):  245 rows
Card 2297 (index):      492 rows
Card 2298 (units):      18 rows
Card 2299 (registry):   58 rows

TUK75 on 2026-03-04: 12 rows  (6 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-04, prev=2026-03-03
  Mgmt fee: 0.205% p.a.

TUK00 on 2026-03-04: 10 rows  (4 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-04, prev=2026-03-03
  Mgmt fee: 0.163% p.a.


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [3]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']
    cash = fd['cash']
    receivables = fd['receivables']
    liabilities = fd['liabilities']
    units_row = fd['units_row']

    sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
    sec_display = sec_display.sort_values('Market Value', ascending=False)
    sec_total = sec_display['Market Value'].sum()

    cash_total = cash['Market Value'].sum()
    recv_total = receivables['Market Value'].sum()
    liab_total = liabilities['Market Value'].sum()
    other_rows = [
        {'Account ID': '', 'Account Name': 'Cash', 'Quantity': None, 'Market Price': None, 'Market Value': cash_total},
        {'Account ID': '', 'Account Name': 'Receivables', 'Quantity': None, 'Market Price': None, 'Market Value': recv_total},
        {'Account ID': '', 'Account Name': 'Liabilities', 'Quantity': None, 'Market Price': None, 'Market Value': liab_total},
    ]

    all_display = pd.concat([sec_display, pd.DataFrame(other_rows)], ignore_index=True)

    # Previous working day comparison
    available_dates = sorted(raw_positions[raw_positions['Fund Code'] == code]['Nav Date'].unique())
    nav_idx = available_dates.index(NAV_DATE) if NAV_DATE in available_dates else -1
    prev_nav_date = available_dates[nav_idx - 1] if nav_idx > 0 else None

    if prev_nav_date:
        prev_pos = raw_positions[
            (raw_positions['Fund Code'] == code) &
            (raw_positions['Nav Date'] == prev_nav_date)
        ]
        prev_sec = prev_pos[prev_pos['Account Type'] == 'SECURITY'].set_index('Account ID')['Market Value']
        prev_cash = prev_pos[prev_pos['Account Type'] == 'CASH']['Market Value'].sum()
        prev_recv = prev_pos[prev_pos['Account Type'] == 'RECEIVABLES']['Market Value'].sum()
        prev_liab = prev_pos[prev_pos['Account Type'] == 'LIABILITY']['Market Value'].sum()
        prev_by_type = {'Cash': prev_cash, 'Receivables': prev_recv, 'Liabilities': prev_liab}

        prev_values = []
        for _, row in all_display.iterrows():
            if row['Account ID'] and row['Account ID'] in prev_sec.index:
                prev_values.append(prev_sec[row['Account ID']])
            elif row['Account Name'] in prev_by_type:
                prev_values.append(prev_by_type[row['Account Name']])
            else:
                prev_values.append(None)

        all_display['Prev Value'] = prev_values
        all_display['Change %'] = (all_display['Market Value'] - all_display['Prev Value']) / all_display['Prev Value'].abs() * 100
    else:
        prev_pos = pd.DataFrame()
        all_display['Prev Value'] = None
        all_display['Change %'] = None

    gross_portfolio = all_display['Market Value'].abs().sum()
    all_display['% of Portfolio'] = all_display['Market Value'] / gross_portfolio * 100

    nav_components = sec_total + cash_total + recv_total + liab_total
    prev_nav_total = all_display['Prev Value'].sum() if prev_nav_date else None
    nav_change_pct = (nav_components - prev_nav_total) / abs(prev_nav_total) * 100 if prev_nav_total else None

    total_row = pd.DataFrame([{
        'Account ID': '', 'Account Name': 'Total net assets', 'Quantity': None,
        'Market Price': None, 'Market Value': nav_components,
        'Prev Value': prev_nav_total, 'Change %': nav_change_pct,
        '% of Portfolio': 100.0
    }])
    all_display = pd.concat([all_display, total_row], ignore_index=True)

    position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

    def bold_total(row):
        if row['Account Name'] == 'Total net assets':
            return ['font-weight: bold'] * len(row)
        return [''] * len(row)

    display(all_display[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value', '% of Portfolio', 'Change %']].style
        .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}',
                 '% of Portfolio': '{:.2f}%', 'Change %': '{:+.2f}%'}, na_rep='')
        .set_caption(f'{code} position as of {NAV_DATE} (vs {prev_nav_date})')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(bold_total, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['sec_total'] = sec_total
    fd['cash_total'] = cash_total
    fd['recv_total'] = recv_total
    fd['liab_total'] = liab_total
    fd['nav_components'] = nav_components
    fd['prev_nav_date'] = prev_nav_date
    fd['prev_nav_total'] = prev_nav_total
    fd['prev_pos'] = prev_pos if prev_nav_date else pd.DataFrame()
    fd['position_units'] = position_units

Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"17,953,577.58",15.4150,"276,754,398.33",29.12%,-0.73%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"8,042,414.77",34.2700,"275,613,554.17",29.00%,-0.73%
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"17,679,728.00",12.1320,"214,490,460.10",22.57%,+1.44%
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"7,550,798.54",13.3080,"100,486,026.97",10.57%,-2.91%
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"7,054,146.00",10.2380,"72,220,346.75",7.60%,+1.86%
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"1,034,931.00",7.6890,"7,957,584.46",0.84%,+3.19%
,Cash,,,"1,646,485.00",0.17%,+0.24%
,Receivables,,,0.00,0.00%,
,Liabilities,,,"-1,122,122.83",-0.12%,-inf%
,Total net assets,,,"948,046,732.95",100.00%,-0.38%


Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
LU0839970364,iShares Global Government Bond Index Fund Class X2,"30,644.79",94.7400,"2,903,287.40",24.61%,-0.05%
IE0031080751,iShares Euro Government Bond Index Fund Flex,"125,765.36",22.8910,"2,878,894.86",24.40%,-0.67%
LU0826455353,iShares Euro Aggregate Bond Index Fund X2,"24,475.69",117.5000,"2,875,893.58",24.38%,+0.25%
IE0005032192,iShares Euro Credit Bond Index Fund Flex,"121,790.70",23.5840,"2,872,311.87",24.35%,-0.44%
,Cash,,,"267,390.47",2.27%,+0.02%
,Receivables,,,0.00,0.00%,
,Liabilities,,,0.00,0.00%,
,Total net assets,,,"11,797,778.18",100.00%,-0.22%


## Price Verification

Cross-checking fund position prices against independent index/provider data. Prices are flagged if the difference exceeds 0.5%.

In [4]:
# ── Key mapping: ISIN → index keys per source ──
KEY_MAP = {
    # TUK75 equity ETFs
    'IE0009FT4LX4': {
        'BlackRock': 'IE0009FT4LX4.BLACKROCK', 'EODHD': 'IE0009FT4LX4.EUFUND',
        'Morningstar': 'IE0009FT4LX4.MORNINGSTAR', 'Yahoo': '0P0001N0Z0.F',
    },
    'IE00BFG1TM61': {
        'BlackRock': 'IE00BFG1TM61.BLACKROCK', 'EODHD': 'IE00BFG1TM61.EUFUND',
        'Morningstar': 'IE00BFG1TM61.MORNINGSTAR', 'Yahoo': '0P000152G5.F',
    },
    'IE00BKPTWY98': {
        'BlackRock': 'IE00BKPTWY98.BLACKROCK', 'EODHD': 'IE00BKPTWY98.EUFUND',
        'Morningstar': 'IE00BKPTWY98.MORNINGSTAR', 'Yahoo': '0P0001MGOG.F',
    },
    'IE00BFNM3G45': {
        'Deutsche Börse': 'IE00BFNM3G45.XETR', 'EODHD': 'SGAS.XETRA', 'Yahoo': 'SGAS.DE',
    },
    'IE00BFNM3D14': {
        'Deutsche Börse': 'IE00BFNM3D14.XETR', 'EODHD': 'SLMC.XETRA', 'Yahoo': 'SLMC.DE',
    },
    'IE00BFNM3L97': {
        'Deutsche Börse': 'IE00BFNM3L97.XETR', 'EODHD': 'SGAJ.XETRA', 'Yahoo': 'SGAJ.DE',
    },
    # TUK00 bond ETFs
    'LU0826455353': {
        'BlackRock': 'LU0826455353.BLACKROCK', 'EODHD': 'LU0826455353.EUFUND',
        'Morningstar': 'LU0826455353.MORNINGSTAR',
    },
    'LU0839970364': {
        'BlackRock': 'LU0839970364.BLACKROCK', 'EODHD': 'LU0839970364.EUFUND',
        'Morningstar': 'LU0839970364.MORNINGSTAR',
    },
    'IE0031080751': {
        'BlackRock': 'IE0031080751.BLACKROCK', 'EODHD': 'IE0031080751.EUFUND',
        'Morningstar': 'IE0031080751.MORNINGSTAR',
    },
    'IE0005032192': {
        'BlackRock': 'IE0005032192.BLACKROCK', 'EODHD': 'IE0005032192.EUFUND',
        'Morningstar': 'IE0005032192.MORNINGSTAR',
    },
}

SOURCES = ['BlackRock', 'Deutsche Börse', 'EODHD', 'Morningstar', 'Yahoo']

# Parse index data
idx = raw_index.copy()
idx['Date'] = pd.to_datetime(idx['Date'])
nav_dt = pd.to_datetime(NAV_DATE)

def latest_price_on_date(key, target_date):
    if key is None:
        return None
    rows = idx[(idx['Key'] == key) & (idx['Date'] == target_date)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]['Value']

def has_any_data_on_date(mapping, target_date):
    """Check if any source has data for this ISIN on the given date."""
    for key in mapping.values():
        if key and latest_price_on_date(key, target_date) is not None:
            return True
    return False

def find_effective_date(isin, mapping, fund_price, base_date):
    """Find the date whose alternative prices match fund_price, starting from base_date."""
    all_keys = [k for k in mapping.values() if k]
    if not all_keys:
        return base_date
    for key in all_keys:
        rows = idx[(idx['Key'] == key) & (idx['Date'] == base_date)]
        if len(rows) > 0 and abs(rows.iloc[0]['Value'] - fund_price) / fund_price * 100 <= 0.01:
            return base_date
    prev_dates = sorted(idx[(idx['Key'].isin(all_keys)) & (idx['Date'] < base_date)]['Date'].unique(), reverse=True)
    for d in prev_dates[:5]:
        for key in all_keys:
            rows = idx[(idx['Key'] == key) & (idx['Date'] == d)]
            if len(rows) > 0 and abs(rows.iloc[0]['Value'] - fund_price) / fund_price * 100 <= 0.01:
                return d
    return base_date

def find_consensus_price(alt_prices, threshold_pct=0.5):
    """Find consensus among ≥2 alternative prices that agree within threshold."""
    prices = [p for p in alt_prices if pd.notna(p)]
    if len(prices) < 2:
        return None
    for i in range(len(prices)):
        agreeing = [prices[i]]
        for j in range(len(prices)):
            if i != j and abs(prices[j] - prices[i]) / prices[i] * 100 <= threshold_pct:
                agreeing.append(prices[j])
        if len(agreeing) >= 2:
            return np.mean(agreeing)
    return None

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']

    verify_rows = []
    for _, sec in securities.sort_values('Market Value', ascending=False).iterrows():
        isin = sec['Account ID']
        mapping = KEY_MAP.get(isin, {})
        fund_price = sec['Market Price']

        # Try NAV date first; if no source has data, assume 1-day lag
        if has_any_data_on_date(mapping, nav_dt):
            base_date = nav_dt
            lag_label = ''
        else:
            base_date = nav_dt - pd.Timedelta(days=1)
            lag_label = ' *'

        eff_date = find_effective_date(isin, mapping, fund_price, base_date)

        row = {
            'ISIN': isin, 'Name': sec['Account Name'] + lag_label, 'Fund': fund_price,
            'Price Date': eff_date.strftime('%Y-%m-%d'),
        }
        for src in SOURCES:
            row[src] = latest_price_on_date(mapping.get(src), eff_date)
        verify_rows.append(row)

    verify_df = pd.DataFrame(verify_rows)

    def highlight_diff(row):
        fund_val = row['Fund']
        styles = [''] * len(row)
        for i, col in enumerate(row.index):
            if col in SOURCES and pd.notna(row[col]):
                diff_pct = abs(row[col] - fund_val) / fund_val * 100
                if diff_pct > 0.5:
                    styles[i] = 'background-color: #f8d7da'
                elif diff_pct > 0.01:
                    styles[i] = 'background-color: #fff3cd'
                else:
                    styles[i] = 'background-color: #d4edda'
        return styles

    has_lagged = any('*' in str(r.get('Name', '')) for r in verify_rows)
    footnote = '  (* = no NAV-date data, using T-1)' if has_lagged else ''
    if len(verify_df) > 0:
        display(verify_df.style
            .format({col: '{:.4f}' for col in ['Fund'] + SOURCES}, na_rep='—')
            .set_caption(f'{code} price verification{footnote}')
            .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
            .apply(highlight_diff, axis=1)
            .hide(axis='index'))

    n_ok = n_flag = n_nodata = 0
    for _, row in verify_df.iterrows():
        for src in SOURCES:
            val = row[src]
            if pd.isna(val) or val is None:
                n_nodata += 1
            elif abs(val - row['Fund']) / row['Fund'] * 100 > 0.5:
                n_flag += 1
            else:
                n_ok += 1

    # Compute repricing adjustment where fund price disagrees with consensus
    reprice_adj = 0.0
    for _, vrow in verify_df.iterrows():
        isin = vrow['ISIN']
        fund_price = vrow['Fund']
        alt_prices = [vrow[src] for src in SOURCES if src in vrow and pd.notna(vrow[src])]
        consensus = find_consensus_price(alt_prices)
        if consensus is not None and abs(consensus - fund_price) / fund_price * 100 > 0.5:
            qty_row = securities[securities['Account ID'] == isin]
            if len(qty_row) > 0:
                reprice_adj += (consensus - fund_price) * qty_row.iloc[0]['Quantity']

    fd['verify_df'] = verify_df
    fd['n_ok'] = n_ok
    fd['n_flag'] = n_flag
    fd['n_nodata'] = n_nodata
    fd['reprice_adj'] = reprice_adj
    if abs(reprice_adj) > 0.01:
        print(f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data  |  reprice adj: {reprice_adj:+,.2f} EUR')
    else:
        print(f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data')

ISIN,Name,Fund,Price Date,BlackRock,Deutsche Börse,EODHD,Morningstar,Yahoo
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0 *,15.4150,2026-03-03,15.4150,—,15.4150,15.4150,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,34.2700,2026-03-03,34.2698,—,34.2700,34.2700,—
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,12.1320,2026-03-04,—,12.1320,12.1320,—,12.1320
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible *,13.3080,2026-03-03,13.3078,—,13.3080,13.3080,—
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,10.2380,2026-03-04,—,10.2380,10.2380,—,10.2380
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,7.6890,2026-03-04,—,7.6890,7.6890,—,7.6890


TUK75: 18 OK  |  0 flagged  |  12 no data


ISIN,Name,Fund,Price Date,BlackRock,Deutsche Börse,EODHD,Morningstar,Yahoo
LU0839970364,iShares Global Government Bond Index Fund Class X2,94.7400,2026-03-04,94.7400,—,94.7400,94.7400,—
IE0031080751,iShares Euro Government Bond Index Fund Flex *,22.8910,2026-03-03,22.8911,—,22.8910,22.8910,—
LU0826455353,iShares Euro Aggregate Bond Index Fund X2,117.5000,2026-03-04,117.5000,—,117.5000,117.5000,—
IE0005032192,iShares Euro Credit Bond Index Fund Flex *,23.5840,2026-03-03,23.5840,—,23.5840,23.5840,—


TUK00: 12 OK  |  0 flagged  |  8 no data


## Net Asset Value Calculation

Computing NAV per unit from position data and comparing to the reported value.

In [5]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    nav_components = fd['nav_components']
    position_units = fd['position_units']
    mgmt_fee_rate = fd['mgmt_fee_rate']
    units_today = fd['units_today']
    prev_nav_date = fd['prev_nav_date']
    prev_nav_total = fd['prev_nav_total']
    prev_pos = fd['prev_pos']
    reprice_adj = fd.get('reprice_adj', 0)
    has_repricing = abs(reprice_adj) > 0.01

    nav_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')
    row_today = units_today.iloc[0]
    reported_nav = row_today['Nav']
    reported_balance = row_today['Balance']

    day_of_month = nav_dt.day
    expected_daily_fee = nav_components * mgmt_fee_rate / 365
    expected_accrual = expected_daily_fee * day_of_month
    nav_after_fees = nav_components - expected_accrual
    computed_nav = nav_after_fees / position_units
    nav_diff_eur = computed_nav - reported_nav
    nav_diff_pct = nav_diff_eur / reported_nav * 100

    # Repriced computation
    if has_repricing:
        repriced_components = nav_components + reprice_adj
        repriced_accrual = repriced_components * mgmt_fee_rate / 365 * day_of_month
        repriced_after_fees = repriced_components - repriced_accrual
        repriced_nav = repriced_after_fees / position_units
        repriced_diff_eur = repriced_nav - reported_nav
        repriced_diff_pct = repriced_diff_eur / reported_nav * 100

    # Previous day
    prev_position_units = None
    prev_accrual = None
    prev_nav_after_fees = None
    prev_computed_nav = None
    if prev_nav_date:
        prev_units_rows = prev_pos[prev_pos['Account Type'] == 'UNITS']
        prev_position_units = prev_units_rows.iloc[0]['Quantity'] if len(prev_units_rows) > 0 else None
        prev_dt = datetime.strptime(prev_nav_date, '%Y-%m-%d')
        prev_accrual = prev_nav_total * mgmt_fee_rate / 365 * prev_dt.day
        prev_nav_after_fees = prev_nav_total - prev_accrual
        if prev_position_units:
            prev_computed_nav = prev_nav_after_fees / prev_position_units

    def fmt_date(d):
        return datetime.strptime(d, '%Y-%m-%d').strftime('%d.%m.%Y') if d else 'Previous'

    def pct_change(cur, prev):
        if cur is not None and prev is not None and prev != 0:
            return f'{(cur - prev) / abs(prev) * 100:+.2f}%'
        return ''

    col_today = fmt_date(NAV_DATE)
    col_prev = fmt_date(prev_nav_date)

    reported_label = 'Total net assets as reported (EUR)' if has_repricing else 'Total net assets (EUR)'
    table_rows = [
        {'': reported_label, col_today: f'{nav_components:,.2f}', col_prev: f'{prev_nav_total:,.2f}' if prev_nav_total else '', 'Change': pct_change(nav_components, prev_nav_total)},
    ]
    if has_repricing:
        table_rows.append(
            {'': 'Total net assets repriced (EUR)', col_today: f'{repriced_components:,.2f}', col_prev: '', 'Change': f'{reprice_adj:+,.2f}'},
        )
    table_rows += [
        {'': 'Accrued mgmt fee est. (EUR)', col_today: f'{expected_accrual:,.2f}', col_prev: f'{prev_accrual:,.2f}' if prev_accrual else '', 'Change': pct_change(expected_accrual, prev_accrual)},
        {'': 'Net assets after fees (EUR)', col_today: f'{nav_after_fees:,.2f}', col_prev: f'{prev_nav_after_fees:,.2f}' if prev_nav_after_fees else '', 'Change': pct_change(nav_after_fees, prev_nav_after_fees)},
        {'': 'Units outstanding', col_today: f'{position_units:,.2f}', col_prev: f'{prev_position_units:,.2f}' if prev_position_units else '', 'Change': pct_change(position_units, prev_position_units)},
        {'': 'Computed NAV (EUR)', col_today: f'{computed_nav:.5f}', col_prev: f'{prev_computed_nav:.5f}' if prev_computed_nav else '', 'Change': pct_change(computed_nav, prev_computed_nav)},
    ]
    if has_repricing:
        table_rows.append(
            {'': 'Computed NAV repriced (EUR)', col_today: f'{repriced_nav:.5f}', col_prev: '', 'Change': f'{repriced_diff_eur:+.5f}'},
        )

    table_df = pd.DataFrame(table_rows)

    ct = col_today
    def style_table(row, col_t=ct):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == col_t:
                styles[i] = 'font-weight: bold'
            elif col_name == 'Change':
                styles[i] = 'font-style: italic'
        return styles

    display(table_df.style
        .set_caption(f'{code} NAV calculation')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(style_table, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['reported_nav'] = reported_nav
    fd['reported_balance'] = reported_balance
    fd['computed_nav'] = computed_nav
    fd['nav_diff_eur'] = nav_diff_eur
    fd['nav_diff_pct'] = nav_diff_pct
    fd['expected_accrual'] = expected_accrual
    fd['total_units'] = position_units
    if has_repricing:
        fd['repriced_nav'] = repriced_nav
        fd['repriced_diff_eur'] = repriced_diff_eur
        fd['repriced_diff_pct'] = repriced_diff_pct

,04.03.2026,03.03.2026,Change
Total net assets (EUR),"948,046,732.95","951,638,328.95",-0.38%
Accrued mgmt fee est. (EUR),"21,298.58","16,034.45",+32.83%
Net assets after fees (EUR),"948,025,434.37","951,622,294.50",-0.38%
Units outstanding,"667,247,468.87","667,244,779.26",+0.00%
Computed NAV (EUR),1.42080,1.42620,-0.38%


,04.03.2026,03.03.2026,Change
Total net assets (EUR),"11,797,778.18","11,824,331.81",-0.22%
Accrued mgmt fee est. (EUR),210.74,158.41,+33.03%
Net assets after fees (EUR),"11,797,567.44","11,824,173.40",-0.23%
Units outstanding,"19,111,960.37","19,111,915.11",+0.00%
Computed NAV (EUR),0.61729,0.61868,-0.23%


## Day-over-Day Comparison

Comparing fund NAV change to a blended benchmark that accounts for pricing lag. Some ETF positions are valued with T-1 prices, so we construct a benchmark mixing T-2→T-1 and T-1→T index changes weighted by the lagged share of the portfolio.

In [6]:
BENCHMARK_SEARCH = {
    'TUK75': ['MSCI_ACWI', 'GLOBAL_STOCK', 'ACWI'],
}

# Share of portfolio priced with T-1 lag (rest is same-day)
LAG_WEIGHT = {'TUK75': 0.70}

# TUK00 custom benchmark: blend of two bond ETF proxies
TUK00_BENCHMARK = {
    'same_day': {'key': 'LU0839970364.BLACKROCK', 'label': 'Global Govt Bond', 'weight': 0.50},
    'lagged':   {'key': 'LU0826455353.BLACKROCK', 'label': 'Euro Agg Bond', 'weight': 0.50},
}

idx_data = raw_index.copy()
idx_data['Date'] = pd.to_datetime(idx_data['Date'])

def clean_series(series):
    """Remove outlier rows where price jumps >20% from neighbours."""
    s = series.sort_values('Date').reset_index(drop=True)
    if len(s) < 3:
        return s
    mask = pd.Series(True, index=s.index)
    for i in range(1, len(s) - 1):
        prev_v, cur_v, next_v = s.loc[i-1, 'Value'], s.loc[i, 'Value'], s.loc[i+1, 'Value']
        if prev_v > 0 and next_v > 0:
            chg_prev = abs(cur_v - prev_v) / prev_v
            chg_next = abs(cur_v - next_v) / next_v
            if chg_prev > 0.20 and chg_next > 0.20:
                mask[i] = False
    # Also check last row against second-to-last
    if len(s) >= 2:
        last_v, prev_v = s.loc[len(s)-1, 'Value'], s.loc[len(s)-2, 'Value']
        if prev_v > 0 and abs(last_v - prev_v) / prev_v > 0.20:
            mask[len(s)-1] = False
    dropped = (~mask).sum()
    if dropped > 0:
        print(f'  ⚠ Dropped {dropped} outlier(s) from price series')
    return s[mask]

def idx_change(key, date_from, date_to):
    """Compute % change of an index key between two dates."""
    series = clean_series(idx_data[idx_data['Key'] == key])
    v_from = series[series['Date'] <= date_from]
    v_to = series[series['Date'] <= date_to]
    if len(v_from) == 0 or len(v_to) == 0:
        return None, None, None
    r_from = v_from.iloc[-1]
    r_to = v_to.iloc[-1]
    if r_from['Date'] == r_to['Date']:
        return 0.0, r_from, r_to
    return (r_to['Value'] - r_from['Value']) / r_from['Value'] * 100, r_from, r_to

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    units_prev = fd['units_prev']
    units_today = fd['units_today']
    prev_date = fd['prev_date']
    reported_nav = fd['reported_nav']

    print(f'═══ {code} ═══')

    if len(units_prev) == 0 or len(units_today) == 0:
        print(f'Cannot compare: missing data for {prev_date} or {NAV_DATE}\n')
        fd['fund_nav_change_pct'] = None
        continue

    prev_nav = units_prev.iloc[0]['Nav']
    today_nav = reported_nav
    fund_nav_change_pct = (today_nav - prev_nav) / prev_nav * 100

    print(f'Fund NAV: {prev_date} {prev_nav:.6f} → {NAV_DATE} {today_nav:.6f}  ({fund_nav_change_pct:+.2f}%)')

    nav_dt_ts = pd.to_datetime(NAV_DATE)
    prev_dt_ts = pd.to_datetime(prev_date)

    if code == 'TUK00':
        # Custom blended benchmark from bond ETF prices
        sd = TUK00_BENCHMARK['same_day']
        lg = TUK00_BENCHMARK['lagged']

        chg_sd, sd_from, sd_to = idx_change(sd['key'], prev_dt_ts, nav_dt_ts)
        # Find T-2 for lagged component (using cleaned series)
        lg_series = clean_series(idx_data[idx_data['Key'] == lg['key']])
        lg_on_prev = lg_series[lg_series['Date'] <= prev_dt_ts]
        if len(lg_on_prev) >= 2:
            lg_t1 = lg_on_prev.iloc[-1]
            lg_t2 = lg_on_prev.iloc[-2]
            chg_lg = (lg_t1['Value'] - lg_t2['Value']) / lg_t2['Value'] * 100
        elif len(lg_on_prev) == 1:
            lg_t1 = lg_on_prev.iloc[-1]
            lg_t2 = lg_t1
            chg_lg = 0.0
        else:
            chg_lg = None

        if chg_sd is not None and chg_lg is not None:
            blended_pct = sd['weight'] * chg_sd + lg['weight'] * chg_lg
            blended_diff = fund_nav_change_pct - blended_pct

            rows = [
                {'Metric': 'Fund NAV change', 'Value': f'{fund_nav_change_pct:+.2f}%'},
                {'Metric': '', 'Value': ''},
                {'Metric': f'{sd["label"]} T-1→T ({sd_from["Date"].strftime("%m-%d")}→{sd_to["Date"].strftime("%m-%d")})', 'Value': f'{chg_sd:+.2f}%'},
                {'Metric': f'{lg["label"]} T-2→T-1 ({lg_t2["Date"].strftime("%m-%d")}→{lg_t1["Date"].strftime("%m-%d")})', 'Value': f'{chg_lg:+.2f}%'},
                {'Metric': f'Blended benchmark ({sd["weight"]:.0%}/{lg["weight"]:.0%})', 'Value': f'{blended_pct:+.2f}%'},
                {'Metric': 'Tracking diff (blended)', 'Value': f'{blended_diff:+.2f}%'},
            ]
            comparison = pd.DataFrame(rows)
            display(comparison.style.hide(axis='index'))

            fd['blended_idx_pct'] = blended_pct
            fd['blended_tracking_diff'] = blended_diff
        else:
            print('Insufficient bond ETF price data for benchmark')

    else:
        # Standard single-index benchmark
        search_terms = BENCHMARK_SEARCH.get(code, [])
        index_keys = [k for k in idx_data['Key'].unique()
                      if any(x in k.upper() for x in search_terms)]

        if index_keys:
            idx_key = index_keys[0]
            idx_series = clean_series(idx_data[idx_data['Key'] == idx_key])

            idx_on_nav = idx_series[idx_series['Date'] <= nav_dt_ts]
            idx_on_prev = idx_series[idx_series['Date'] <= prev_dt_ts]

            if len(idx_on_nav) > 0 and len(idx_on_prev) > 0:
                idx_today_row = idx_on_nav.iloc[-1]
                idx_prev_row = idx_on_prev.iloc[-1]

                if idx_today_row['Date'] != idx_prev_row['Date']:
                    idx_change_pct = (idx_today_row['Value'] - idx_prev_row['Value']) / idx_prev_row['Value'] * 100
                    tracking_diff = fund_nav_change_pct - idx_change_pct
                else:
                    idx_change_pct = 0.0
                    tracking_diff = fund_nav_change_pct

                rows = [
                    {'Metric': 'Fund NAV change', 'Value': f'{fund_nav_change_pct:+.2f}%'},
                    {'Metric': f'Index change ({idx_key})', 'Value': f'{idx_change_pct:+.2f}%'},
                    {'Metric': 'Tracking difference', 'Value': f'{tracking_diff:+.2f}%'},
                ]

                # Blended index: account for lagged pricing
                lag_w = LAG_WEIGHT.get(code, 0)
                if lag_w > 0 and idx_today_row['Date'] != idx_prev_row['Date']:
                    idx_before_prev = idx_series[idx_series['Date'] < prev_dt_ts]
                    if len(idx_before_prev) > 0:
                        idx_t2_row = idx_before_prev.iloc[-1]
                        chg_t2_t1 = (idx_prev_row['Value'] - idx_t2_row['Value']) / idx_t2_row['Value'] * 100
                        chg_t1_t0 = idx_change_pct
                        blended_pct = lag_w * chg_t2_t1 + (1 - lag_w) * chg_t1_t0
                        blended_diff = fund_nav_change_pct - blended_pct

                        rows += [
                            {'Metric': '', 'Value': ''},
                            {'Metric': f'Index T-2→T-1 ({idx_t2_row["Date"].strftime("%m-%d")}→{idx_prev_row["Date"].strftime("%m-%d")})', 'Value': f'{chg_t2_t1:+.2f}%'},
                            {'Metric': f'Index T-1→T ({idx_prev_row["Date"].strftime("%m-%d")}→{idx_today_row["Date"].strftime("%m-%d")})', 'Value': f'{chg_t1_t0:+.2f}%'},
                            {'Metric': f'Blended index ({lag_w:.0%} lagged)', 'Value': f'{blended_pct:+.2f}%'},
                            {'Metric': 'Tracking diff (blended)', 'Value': f'{blended_diff:+.2f}%'},
                        ]

                        fd['blended_idx_pct'] = blended_pct
                        fd['blended_tracking_diff'] = blended_diff

                comparison = pd.DataFrame(rows)
                display(comparison.style.hide(axis='index'))

                fd['idx_key'] = idx_key
                fd['idx_change_pct'] = idx_change_pct
                fd['tracking_diff'] = tracking_diff
            else:
                print('Index data insufficient for comparison')
        else:
            print('No benchmark index found')

    fd['fund_nav_change_pct'] = fund_nav_change_pct
    print()

═══ TUK75 ═══
Fund NAV: 2026-03-03 1.427770 → 2026-03-04 1.426200  (-0.11%)


Metric,Value
Fund NAV change,-0.11%
Index change (MSCI_ACWI),-0.39%
Tracking difference,+0.28%
,
Index T-2→T-1 (03-02→03-03),-1.00%
Index T-1→T (03-03→03-04),-0.39%
Blended index (70% lagged),-0.82%
Tracking diff (blended),+0.71%



═══ TUK00 ═══
Fund NAV: 2026-03-03 0.620440 → 2026-03-04 0.618680  (-0.28%)
  ⚠ Dropped 1 outlier(s) from price series
  ⚠ Dropped 1 outlier(s) from price series


Metric,Value
Fund NAV change,-0.28%
,
Global Govt Bond T-1→T (03-03→03-04),-0.05%
Euro Agg Bond T-2→T-1 (03-02→03-03),-0.65%
Blended benchmark (50%/50%),-0.35%
Tracking diff (blended),+0.07%
